# Candidatos para dashboard

Este cuaderno compara visualizaciones candidatas para decidir que consultas merecen un hueco en el dashboard final.

Regla practica:
- KPI para resumir estado general.
- Ranking para priorizar focos.
- Serie temporal para detectar picos y regresiones.
- Tabla detalle para investigacion, no para portada.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd()
candidate_roots = [
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == 'notebooks' else NOTEBOOK_CWD,
    NOTEBOOK_CWD / 'observability' / 'd365-fo-observability',
]
PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'kql_runner.py').exists() and (candidate / 'queries').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('No se pudo localizar observability/d365-fo-observability desde el directorio actual')
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from kql_runner import build_client, load_config, load_kql, metric_snapshot, plot_bar, plot_timeseries, run_kql, set_analyst_theme, summarize_top_items

config = load_config()
client = build_client(config=config)
DASHBOARD_DAYS = config['query_days']
TOP_N = 12
set_analyst_theme()

In [ ]:
df_sli = run_kql(client, load_kql('10_sli_executive_summary.kql'), days=DASHBOARD_DAYS, name='SLI executive summary', config=config)
display(df_sli)
if not df_sli.empty:
    display(metric_snapshot({
        'Form loads': int(df_sli.loc[0, 'FormLoads']),
        'P95 form load (s)': float(df_sli.loc[0, 'P95FormLoadSeconds']),
        'Exceptions': int(df_sli.loc[0, 'Exceptions']),
        'Slow queries': int(df_sli.loc[0, 'SlowQueries']),
    }))

In [ ]:
df_forms = run_kql(client, load_kql('22_forms_total_impact.kql'), days=DASHBOARD_DAYS, name='Forms total impact', config=config)
display(df_forms.head(TOP_N))
display(summarize_top_items(df_forms, 'FormName', 'TotalDurationMinutes', top_n=5))
plot_bar(df_forms, 'FormName', 'TotalDurationMinutes', 'Candidato dashboard: formularios con mayor impacto total', top_n=TOP_N)

In [ ]:
df_exception_series = run_kql(client, load_kql('41_exceptions_timeseries.kql'), days=DASHBOARD_DAYS, name='Exceptions timeseries', config=config)
display(df_exception_series.tail(24))
plot_timeseries(df_exception_series, 'timestamp', ['Exceptions'], 'Candidato dashboard: evolucion de excepciones')

In [ ]:
df_availability = run_kql(client, load_kql('60_availability.kql'), days=max(DASHBOARD_DAYS, 30), name='Availability', config=config)
display(df_availability)
plot_bar(df_availability, 'name', 'AvailabilityPercent', 'Candidato dashboard: disponibilidad por test', top_n=TOP_N, ascending=True)

## Checklist de decision

- La audiencia entiende la metrica sin conocer la consulta KQL?
- La visualizacion ayuda a decidir o solo a describir?
- El dato es estable y accionable?
- Existe suficiente volumen para que el widget no sea ruido?